[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [47]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.rank=rank
        self.alpha=alpha
        self.linear=nn.Linear(in_features,out_features)
        self.linear.weight.requires_grad=False
        self.linear.bias.requires_grad=False
        self.lora_A=nn.Parameter(torch.randn(rank,in_features))
        self.lora_B=nn.Parameter(torch.zeros(out_features,rank))

        pass  # frozen linear + lora_A + lora_B

    def forward(self, x):
        down_projected=torch.matmul(x,self.lora_A.T)
        up_projected=torch.matmul(down_projected,self.lora_B.T)
        self.scaling=self.alpha/self.rank

        out =self.linear(x)+self.scaling*up_projected
        return out
        pass  # base + lora

In [48]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [49]:
# ✅ SUBMIT
from torch_judge import check,hint
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (0.8ms)
  ✅ [2/5] LoRA parameter shapes (0.4ms)
  ✅ [3/5] B=0 means output equals base (3.5ms)
  ✅ [4/5] Only LoRA params get gradients (0.8ms)
  ✅ [5/5] Forward computation (1.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (6.9ms total)
  Progress saved. Run status() to see your dashboard.



In [41]:
hint("lora")


💡 Hint for LoRA (Low-Rank Adaptation):
   Freeze base linear. Add lora_A (rank, in) and lora_B (out, rank) as Parameters. B init to zeros. output = linear(x) + (x @ A^T @ B^T) * (alpha/rank).



You csnnot directly call nn.PAatemer as nn.Linear(X) it has to be matrix multiplication. cannot do lora_B(x)